# 26 Normalisasi Teks Informal

Mengubah teks informal menjadi bahasa baku. Contoh: dataset STIF-Indonesia, dari kicauan layanan pelanggan.

**Daftar isi**

1. Kartu Masalah
2. Kapan Pakai Notebook Ini
3. Config
4. EDA
5. Preprocessing
6. Validasi
7. Baseline
8. Model Utama
9. Post-processing
10. Post-analysis
11. Keluaran

## 1. Kartu Masalah

ISI_SETELAH_EDA

## 2. Kapan Pakai Notebook Ini

**Pakai kalau** soal meminta menormalkan slang, atau sebagai langkah awal untuk kata kunci dan topik. Untuk klasifikasi
dengan IndoBERTweet biasanya tidak perlu, karena model itu sudah terbiasa dengan teks informal.

**Yang perlu diganti kalau datanya beda:**
- `read_pair`, kalau pasangan teks disimpan dengan format lain.
- Kamus di `load_slang_dict`, tambahkan istilah domain, misalnya `sppg` dan `bgn` yang tidak perlu diubah.

## 3. Config

**Import dan lingkungan.** Mode offline dinyalakan di sini, dan progress bar serta warning
dimatikan supaya output tetap bersih.

In [1]:
import importlib.util
import json
import os
import random
import subprocess
import sys
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

os.environ.update(
    CUDA_VISIBLE_DEVICES="0",
    HF_HUB_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
    TOKENIZERS_PARALLELISM="false",
    TQDM_DISABLE="1",
    PYTHONWARNINGS="ignore",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)
warnings.filterwarnings("ignore")
sys.modules["tensorflow"] = None  # tidak dipakai; mencegah log TensorFlow saat umap diimpor

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 90, "display.max_rows", 30, "display.width", 140)
plt.rcParams.update(
    {
        "figure.figsize": (8, 3.5),
        "figure.dpi": 90,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print(f"device={DEVICE} | torch={torch.__version__}")

device=cuda | torch=2.11.0+cu130


**Lokasi file.** `find_pack` mencari paket offline, dan `find_file` mencari file data di Kaggle
atau folder lokal. Di mesin lokal, set `PACK_DIR` dan `DATA_ROOT` lewat environment variable.

In [2]:
INPUT_ROOTS = [Path(p) for p in (os.environ.get("DATA_ROOT"), "/kaggle/input", ".") if p]


def find_pack() -> Path:
    """Cari folder paket offline lewat file pack_manifest.json."""
    if os.environ.get("PACK_DIR"):
        return Path(os.environ["PACK_DIR"])
    for root in INPUT_ROOTS:
        for manifest in root.glob("**/pack_manifest.json"):
            return manifest.parent
    raise FileNotFoundError("Paket offline tidak ketemu. Set environment variable PACK_DIR.")


def find_file(name: str) -> Path:
    """Cari file data berdasarkan nama di semua folder input."""
    for root in INPUT_ROOTS:
        for path in root.glob(f"**/{name}"):
            return path
    raise FileNotFoundError(f"File {name} tidak ketemu di {INPUT_ROOTS}")


def install_offline(*modules: str) -> None:
    """Pasang library dari wheel paket offline, hanya bila belum terpasang."""
    missing = [m for m in modules if importlib.util.find_spec(m.replace("-", "_")) is None]
    if missing:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-index",
                "--find-links",
                str(PACK_DIR / "wheels"),
                "--root-user-action=ignore",
                *missing,
            ],
            check=True,
        )


PACK_DIR = find_pack()
MODEL_DIR = PACK_DIR / "models"
EXTRA_DATA_DIR = PACK_DIR / "data"
RESULT_DIR = Path("hasil")
RESULT_DIR.mkdir(exist_ok=True)
print(f"paket offline: {PACK_DIR}")

paket offline: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline


**Pencatat waktu dan ringkasan.** `stage` mencatat durasi tiap tahap, dan `run_summary` mencetak
ringkasan akhir yang dibaca skrip audit.

In [3]:
RUN_START = time.time()
STAGE_SECONDS: dict[str, float] = {}


@contextmanager
def stage(name: str):
    """Catat durasi satu tahap."""
    start = time.time()
    yield
    STAGE_SECONDS[name] = round(time.time() - start, 1)
    print(f"[waktu] {name}: {STAGE_SECONDS[name]:.0f} detik")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_summary(task: str, n_rows: int, metrics: dict, needs_gpu: bool = True) -> None:
    """Simpan dan cetak ringkasan run."""
    total = int(time.time() - RUN_START)
    used_gpu = torch.cuda.is_available() and torch.cuda.max_memory_allocated() > 0
    summary = {
        "task": task,
        "mode": MODE,
        "use_pretrained": USE_PRETRAINED,
        "n_rows": n_rows,
        "total_detik": total,
        "gpu": used_gpu,
        "tahap_detik": STAGE_SECONDS,
        "metrik": metrics,
    }
    (RESULT_DIR / "ringkasan.json").write_text(json.dumps(summary, indent=1))
    note = "" if needs_gpu else " gpu_tidak_wajib"
    print(
        f"RINGKASAN_RUN task={task} mode={MODE} n_rows={n_rows} total_detik={total} "
        f"gpu={used_gpu}{note}"
    )
    print(json.dumps(metrics, indent=1))

**Pengaturan utama.** Semua yang perlu diubah ada di sel ini.

In [4]:
MODE = "penuh"  # "cepat" (<30 menit) atau "penuh" (<2 jam)
USE_PRETRAINED = True  # False: hanya kamus, tanpa file model dari luar
SEED = 42
N_ROWS = 5000
PREFIX = "normalisasi: "

SETTINGS = {"cepat": {"epochs": 2}, "penuh": {"epochs": 8}}[MODE]

seed_everything(SEED)
install_offline("sacrebleu", "jiwer")
print(SETTINGS)

{'epochs': 8}


**Util teks.** `clean_text` hanya membuang gangguan teknis (URL, mention, spasi) dan
mempertahankan emoji serta tanda seru karena keduanya membawa sinyal. `normalize_slang` memakai
kamus alay. `near_duplicate_groups` menemukan komentar kembar supaya tidak bocor antar fold.

In [5]:
import re  # noqa: E402

from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import roc_auc_score  # noqa: E402
from sklearn.model_selection import cross_val_predict  # noqa: E402
from sklearn.neighbors import NearestNeighbors  # noqa: E402

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Bersihkan URL, mention, dan spasi berlebih. Huruf kecil, emoji tetap."""
    text = URL_RE.sub(" ", str(text))
    text = MENTION_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip().lower()


def load_slang_dict() -> dict[str, str]:
    """Muat kamus kata informal ke kata baku."""
    path = next((EXTRA_DATA_DIR / "kamus_alay").glob("**/colloquial-indonesian-lexicon.csv"))
    lexicon = pd.read_csv(path)
    return dict(zip(lexicon["slang"].astype(str), lexicon["formal"].astype(str)))


def normalize_slang(text: str, slang: dict[str, str]) -> str:
    """Ganti kata informal dengan kata baku, per kata."""
    return " ".join(slang.get(word, word) for word in text.split())


def near_duplicate_groups(texts: list[str], threshold: float = 0.9) -> np.ndarray:
    """Beri id grup yang sama pada teks yang kemiripan char TF-IDF-nya di atas threshold."""
    vectors = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(texts)
    neighbors = NearestNeighbors(n_neighbors=min(6, len(texts)), metric="cosine").fit(vectors)
    distance, index = neighbors.kneighbors(vectors)
    parent = list(range(len(texts)))

    def root(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, (row_dist, row_idx) in enumerate(zip(distance, index)):
        for dist, j in zip(row_dist, row_idx):
            if dist <= 1 - threshold:
                parent[root(i)] = root(int(j))
    return np.array([root(i) for i in range(len(texts))])


def adversarial_auc(texts_a: list[str], texts_b: list[str]) -> float:
    """AUC model yang membedakan dua kumpulan teks. Dekat 0,5 berarti tidak ada shift."""
    texts = list(texts_a) + list(texts_b)
    target = np.array([0] * len(texts_a) + [1] * len(texts_b))
    vectors = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    scores = cross_val_predict(model, vectors, target, cv=3, method="predict_proba")[:, 1]
    return float(roc_auc_score(target, scores))


def text_profile(texts: pd.Series) -> pd.Series:
    """Ringkasan kualitas teks untuk EDA."""
    words = texts.astype(str).str.split().str.len()
    return pd.Series(
        {
            "baris": len(texts),
            "kosong": int((words == 0).sum()),
            "duplikat_persis_%": round(100 * texts.duplicated().mean(), 1),
            "kata_median": float(words.median()),
            "kata_p95": float(words.quantile(0.95)),
            "kata_maks": int(words.max()),
            "ada_url_%": round(100 * texts.astype(str).str.contains(r"https?://").mean(), 1),
            "ada_emoji_%": round(
                100 * texts.astype(str).str.contains(r"[\U0001F300-\U0001FAFF]").mean(), 1
            ),
        }
    )

**Seq2seq.** `generate_texts` menghasilkan teks dari model pretrained. `finetune_seq2seq` melatih model
itu pada data lu sendiri. Bobot akhir tidak disimpan, supaya folder hasil tetap kecil.

In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as SeqTokenizer  # noqa: E402


def load_seq2seq(name: str):
    tokenizer = SeqTokenizer.from_pretrained(MODEL_DIR / name)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR / name).to(DEVICE)
    return tokenizer, model


@torch.inference_mode()
def generate_texts(
    model, tokenizer, texts, max_input=256, max_new=64, beams=4, batch_size=32, prefix=""
):
    """Hasilkan teks, urut dari yang terpanjang supaya padding lebih hemat."""
    model.eval()
    order = np.argsort([-len(t) for t in texts])
    outputs = [""] * len(texts)
    for start in range(0, len(texts), batch_size):
        picks = order[start : start + batch_size]
        tokens = tokenizer(
            [prefix + texts[i] for i in picks],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input,
        ).to(DEVICE)
        generated = model.generate(
            **tokens, max_new_tokens=max_new, num_beams=beams, no_repeat_ngram_size=3
        )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        for i, text in zip(picks, decoded):
            outputs[i] = text.strip()
    return outputs


def finetune_seq2seq(
    model,
    tokenizer,
    sources,
    targets,
    epochs,
    seed,
    lr=3e-5,
    batch_size=16,
    max_input=256,
    max_target=96,
    prefix="",
):
    """Latih model seq2seq dengan loss bawaan."""
    seed_everything(seed)
    source_tokens = tokenizer(
        [prefix + s for s in sources],
        truncation=True,
        max_length=max_input,
        padding="max_length",
        return_tensors="pt",
    )
    target_ids = tokenizer(
        text_target=list(targets),
        truncation=True,
        max_length=max_target,
        padding="max_length",
        return_tensors="pt",
    )["input_ids"]
    target_ids[target_ids == tokenizer.pad_token_id] = -100
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    generator = np.random.default_rng(seed)
    torch.cuda.empty_cache()
    model.train()
    for _ in range(epochs):
        order = generator.permutation(len(sources))
        for start in range(0, len(order), batch_size):
            pick = torch.as_tensor(order[start : start + batch_size])
            optimizer.zero_grad(set_to_none=True)
            # Potong padding per batch: logit mT5 sangat besar karena kosakatanya 250 ribu.
            attention = source_tokens["attention_mask"][pick]
            source_len = int(attention.sum(1).max())
            target_len = int((target_ids[pick] != -100).sum(1).max())
            loss = model(
                input_ids=source_tokens["input_ids"][pick][:, :source_len].to(DEVICE),
                attention_mask=attention[:, :source_len].to(DEVICE),
                labels=target_ids[pick][:, :target_len].to(DEVICE),
            ).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return model

## 4. EDA

Muat pasangan informal dan baku. File `.inf` berisi teks informal, dan `.for` berisi teks baku.

In [7]:
import jiwer  # noqa: E402
import sacrebleu  # noqa: E402


def read_pair(split: str) -> pd.DataFrame:
    folder = EXTRA_DATA_DIR / "stif/data/labelled"
    informal = (folder / f"{split}.inf").read_text(encoding="utf-8").splitlines()
    formal = (folder / f"{split}.for").read_text(encoding="utf-8").splitlines()
    return pd.DataFrame({"informal": informal, "formal": formal}).head(N_ROWS)


train, valid, test = read_pair("train"), read_pair("dev"), read_pair("test")
print(f"latih={len(train)} | valid={len(valid)} | test={len(test)}")
print("informal:", train["informal"][0][:100])
print("baku:", train["formal"][0][:100])
same = (train["informal"] == train["formal"]).mean()
print(f"pasangan identik: {same:.1%}")
text_profile(train["informal"])

latih=1922 | valid=214 | test=363
informal: alhamdulillah stlh libur xxxnumberxxx hari onbid lgsg dikasih orderan , food lg . thanks xxxuserxxx 
baku: alhamdulillah setelah libur xxxnumberxxx hari onbid langsung diberi order , makanan lagi . terima ka
pasangan identik: 1.5%


baris                1922.0
kosong                  0.0
duplikat_persis_%       0.0
kata_median            14.0
kata_p95               26.0
kata_maks              43.0
ada_url_%               0.0
ada_emoji_%             2.2
dtype: float64

Kata yang paling sering berubah, dan seberapa banyak kata yang berubah per kalimat.

In [8]:
from collections import Counter  # noqa: E402

changes: Counter = Counter()
for informal, formal in zip(train["informal"], train["formal"]):
    a, b = informal.split(), formal.split()
    if len(a) == len(b):
        changes.update((x, y) for x, y in zip(a, b) if x != y)
print("perubahan tersering:", changes.most_common(12))
print(
    "CER jika teks tidak diubah:",
    round(jiwer.cer(train["formal"].tolist(), train["informal"].tolist()), 4),
)

perubahan tersering: [(('min', 'admin'), 74), ((',', '.'), 44), (('ga', 'tidak'), 32), (('udah', 'sudah'), 25), (('yg', 'yang'), 22), (('gak', 'tidak'), 21), (('ya', '?'), 18), (('ya', '.'), 17), (('sy', 'saya'), 15), (('nya', '.'), 12), (('.', 'terima'), 11), (('gimana', 'bagaimana'), 10)]
CER jika teks tidak diubah: 0.228


**Temuan dan keputusan**

ISI_SETELAH_EDA

## 5. Preprocessing

Token seperti `xxxuserxxx` dan `xxxnumberxxx` adalah hasil anonimisasi dan harus dipertahankan persis. Tidak ada
pembersihan lain, karena targetnya sendiri menentukan bentuk akhir.

In [9]:
PLACEHOLDER_RE = re.compile(r"xxx\w+xxx")
print(
    "placeholder di latih:",
    Counter(PLACEHOLDER_RE.findall(" ".join(train["informal"]))).most_common(5),
)

placeholder di latih: [('xxxnumberxxx', 807), ('xxxuserxxx', 318), ('xxxphonexxx', 36), ('xxxdatexxx', 15), ('xxxpercentxxx', 14)]


## 6. Validasi

Memakai pembagian resmi. Metrik: CER dan WER (makin kecil makin baik) serta BLEU. Baseline salin teks wajib ada,
karena pada tugas ini model sering justru memperburuk teks yang sudah benar.

In [10]:
def normalization_scores(predictions: list[str], references: list[str]) -> dict[str, float]:
    return {
        "cer": round(jiwer.cer(references, predictions), 4),
        "wer": round(jiwer.wer(references, predictions), 4),
        "bleu": round(sacrebleu.corpus_bleu(predictions, [references], force=True).score, 2),
    }


assert normalization_scores(["a b c d"], ["a b c d"])["cer"] == 0.0
results: list[dict] = []


def record(name: str, pred_valid: list[str], pred_test: list[str]) -> None:
    valid_scores = normalization_scores(pred_valid, valid["formal"].tolist())
    test_scores = normalization_scores(pred_test, test["formal"].tolist())
    results.append(
        {
            "metode": name,
            "cer_valid": valid_scores["cer"],
            "cer_test": test_scores["cer"],
            "wer_test": test_scores["wer"],
            "bleu_test": test_scores["bleu"],
        }
    )
    print(results[-1])

## 7. Baseline

Salin teks, lalu kamus alay, lalu kamus dari data latih itu sendiri yang biasanya lebih cocok dengan domainnya.

In [11]:
record("salin_teks", valid["informal"].tolist(), test["informal"].tolist())
slang = load_slang_dict()
record(
    "kamus_alay",
    [normalize_slang(t, slang) for t in valid["informal"]],
    [normalize_slang(t, slang) for t in test["informal"]],
)

learned: dict[str, Counter] = {}
for (word_from, word_to), count in changes.items():
    learned.setdefault(word_from, Counter())[word_to] += count
keep_counts = Counter(w for t in train["formal"] for w in t.split())
learned_map = {
    w: c.most_common(1)[0][0]
    for w, c in learned.items()
    if c.most_common(1)[0][1] >= 2 and c.most_common(1)[0][1] > keep_counts[w]
}
combined = {**slang, **learned_map}
valid_dict = [normalize_slang(t, combined) for t in valid["informal"]]
test_dict = [normalize_slang(t, combined) for t in test["informal"]]
record("kamus_gabungan", valid_dict, test_dict)

{'metode': 'salin_teks', 'cer_valid': 0.22, 'cer_test': 0.2299, 'wer_test': 0.4445, 'bleu_test': 32.51}


{'metode': 'kamus_alay', 'cer_valid': 0.1987, 'cer_test': 0.2087, 'wer_test': 0.384, 'bleu_test': 40.06}
{'metode': 'kamus_gabungan', 'cer_valid': 0.1807, 'cer_test': 0.1932, 'wer_test': 0.3507, 'bleu_test': 43.23}


## 8. Model Utama

mT5 dilatih memetakan informal ke baku. Model pretrained ini aslinya untuk meringkas, jadi fine-tune wajib.

In [12]:
valid_model, test_model = valid_dict, test_dict
if USE_PRETRAINED:
    tokenizer, model = load_seq2seq("mt5_sum")
    with stage("mt5_finetune"):
        model = finetune_seq2seq(
            model,
            tokenizer,
            train["informal"].tolist(),
            train["formal"].tolist(),
            SETTINGS["epochs"],
            SEED,
            batch_size=8,
            max_input=96,
            max_target=96,
            prefix=PREFIX,
            lr=1e-4,
        )
    with stage("mt5_generate"):
        valid_model = generate_texts(
            model,
            tokenizer,
            valid["informal"].tolist(),
            max_input=96,
            max_new=96,
            prefix=PREFIX,
            beams=2,
        )
        test_model = generate_texts(
            model,
            tokenizer,
            test["informal"].tolist(),
            max_input=96,
            max_new=96,
            prefix=PREFIX,
            beams=2,
        )
    record("mt5_finetune", valid_model, test_model)
    del model
    torch.cuda.empty_cache()

[waktu] mt5_finetune: 1101 detik


[waktu] mt5_generate: 27 detik
{'metode': 'mt5_finetune', 'cer_valid': 0.1643, 'cer_test': 0.1712, 'wer_test': 0.2862, 'bleu_test': 50.48}


## 9. Post-processing

**Pengaman.** Keluaran model yang panjangnya jauh berbeda dari masukan, atau kehilangan placeholder, dianggap
gagal dan diganti hasil kamus. Pemilihan akhir antara kamus dan model memakai CER di valid.

In [13]:
def guarded(source: str, predicted: str, fallback: str) -> str:
    ratio = len(predicted.split()) / max(len(source.split()), 1)
    placeholders_ok = sorted(PLACEHOLDER_RE.findall(source)) == sorted(
        PLACEHOLDER_RE.findall(predicted)
    )
    return predicted if 0.6 <= ratio <= 1.6 and placeholders_ok else fallback


valid_guard = [guarded(s, p, f) for s, p, f in zip(valid["informal"], valid_model, valid_dict)]
test_guard = [guarded(s, p, f) for s, p, f in zip(test["informal"], test_model, test_dict)]
record("model_plus_pengaman", valid_guard, test_guard)
use_model = jiwer.cer(valid["formal"].tolist(), valid_guard) < jiwer.cer(
    valid["formal"].tolist(), valid_dict
)
final_test = test_guard if use_model else test_dict
print("dipilih dari valid:", "model dengan pengaman" if use_model else "kamus gabungan")

{'metode': 'model_plus_pengaman', 'cer_valid': 0.1589, 'cer_test': 0.166, 'wer_test': 0.2828, 'bleu_test': 51.08}
dipilih dari valid: model dengan pengaman


## 10. Post-analysis

**Tabel perbandingan.**

In [14]:
result_table = pd.DataFrame(results)
result_table.to_csv(RESULT_DIR / "perbandingan.csv", index=False)
result_table

,metode,cer_valid,cer_test,wer_test,bleu_test
0,salin_teks,0.2200,0.2299,0.4445,32.51
1,kamus_alay,0.1987,0.2087,0.3840,40.06
2,kamus_gabungan,0.1807,0.1932,0.3507,43.23
3,mt5_finetune,0.1643,0.1712,0.2862,50.48
4,model_plus_pengaman,0.1589,0.1660,0.2828,51.08


**Contoh hasil,** diurutkan dari kalimat dengan selisih terbesar.

In [15]:
examples = pd.DataFrame(
    {
        "informal": test["informal"].str[:60],
        "baku": test["formal"].str[:60],
        "hasil": [t[:60] for t in final_test],
    }
)
examples["cer"] = [
    round(jiwer.cer(r, p), 3) if p else 1.0 for r, p in zip(test["formal"], final_test)
]
examples.sort_values("cer", ascending=False).head(6)

,informal,baku,hasil,cer
357,diapk gojek ga ada gohug ? cause i really really need it ),di aplikasi gojek tidak ada gohug ?,diap gojek tidak ada gohug ? sebab saya really really need i,1.200
4,tolong dong udah xxxnumberxxx xxxnumberxxx x kayagini . kena,"tolong ! sudah seperti ini , kenapa ?",tolong dong sudah xxxnumberxxx xxxnumberxxx kali kayagini .,1.054
66,ah elu put cek aje di hot offer,"kamu , put , cek saja di penawaran menarik .","ah , kamu put cek aje di hot offer .",0.591
275,biar dapet birthday gift gimana sii . pengen dong,agar dapat hadiah ulang tahun bagaimanakah ? saya mau .,supaya dapat birthday gift bagaimana ? ingin dong .,0.582
325,ayo dong aku blm dapet ni,"ayo , saya belum dapat .","ayo , aku belum mendapatkan ini .",0.542
328,virtual acc bca lagi eror min ?,"akun virtual bca sedang bersalahkah , admin ?",virtual account bca sedang error admin ?,0.511


## 11. Keluaran

File untuk dikumpulkan dan ringkasan run.

In [16]:
pd.DataFrame({"id": test.index, "baku": final_test}).to_csv(
    RESULT_DIR / "submission.csv", index=False
)
run_summary(
    "normalisasi",
    len(train),
    {r["metode"]: {"cer_test": r["cer_test"], "bleu_test": r["bleu_test"]} for r in results},
    needs_gpu=USE_PRETRAINED,
)

RINGKASAN_RUN task=normalisasi mode=penuh n_rows=1922 total_detik=1607 gpu=True
{
 "salin_teks": {
  "cer_test": 0.2299,
  "bleu_test": 32.51
 },
 "kamus_alay": {
  "cer_test": 0.2087,
  "bleu_test": 40.06
 },
 "kamus_gabungan": {
  "cer_test": 0.1932,
  "bleu_test": 43.23
 },
 "mt5_finetune": {
  "cer_test": 0.1712,
  "bleu_test": 50.48
 },
 "model_plus_pengaman": {
  "cer_test": 0.166,
  "bleu_test": 51.08
 }
}


**Poin untuk presentasi**

ISI_SETELAH_EDA